<a href="https://colab.research.google.com/github/matiasfortegar/GO_APP/blob/main/AUTOMATIZACION_PDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. INSTALACIÓN DE DEPENDENCIAS
!apt-get install -y tesseract-ocr
!apt-get install -y poppler-utils
!pip install pytesseract pdf2image pymupdf

import os
import pytesseract
from pdf2image import convert_from_path
import fitz  # PyMuPDF
from google.colab import files
import shutil
import time

def preparar_ambiente():
    for carpeta in ['input', 'output']:
        if os.path.exists(carpeta):
            shutil.rmtree(carpeta)
        os.makedirs(carpeta)
    print("✅ Entorno listo.")

def ocr_pdf(ruta_entrada, ruta_salida):
    try:
        paginas = convert_from_path(ruta_entrada, 200)
        pdf_data = pytesseract.image_to_pdf_or_hocr(paginas[0], extension='pdf')
        with open(ruta_salida, 'wb') as f:
            f.write(pdf_data)
    except Exception as e:
        print(f"❌ Error OCR: {e}")

def reparar_pdf(ruta_entrada, ruta_salida):
    try:
        doc = fitz.open(ruta_entrada)
        doc.save(ruta_salida, garbage=4, deflate=True, clean=True)
        doc.close()
    except Exception as e:
        print(f"❌ Error Reparar: {e}")

def comprimir_pdf(ruta_entrada, ruta_salida):
    try:
        doc = fitz.open(ruta_entrada)
        doc.save(ruta_salida, garbage=4, deflate=True)
        doc.close()
    except Exception as e:
        print(f"❌ Error Comprimir: {e}")

# --- FLUJO PRINCIPAL ---

def ejecutar_automatizacion_personalizada():
    preparar_ambiente()

    print("Sube tus 2 archivos (Original y MS Print):")
    subidos = files.upload()
    nombres_reales = list(subidos.keys())

    if len(nombres_reales) < 2:
        print("⚠️ Error: Se necesitan 2 archivos.")
        return

    # Extraer nombres base sin extensión para el prefijo
    # Ejemplo: "Factura_01.pdf" -> "Factura_01"
    base_nom1 = os.path.splitext(nombres_reales[0])[0]
    base_nom2 = os.path.splitext(nombres_reales[1])[0]

    v = {}
    # Nivel 1: Entradas originales (Mantenemos nombre original en carpeta input)
    v[1] = os.path.join('input', nombres_reales[0])
    v[2] = os.path.join('input', nombres_reales[1])
    os.rename(nombres_reales[0], v[1])
    os.rename(nombres_reales[1], v[2])

    # Nivel 2: OCR
    v[3] = f"output/{base_nom1}_V03_OCR.pdf"
    v[4] = f"output/{base_nom2}_V04_OCR.pdf"
    print(f"--- Procesando OCR...")
    ocr_pdf(v[1], v[3])
    ocr_pdf(v[2], v[4])

    # Nivel 3: Reparados (V5 a V8 basados en V1-V4)
    print(f"--- Reparando documentos...")
    nombres_nivel_previo = [base_nom1, base_nom2, f"{base_nom1}_OCR", f"{base_nom2}_OCR"]
    for i in range(1, 5):
        v[i+4] = f"output/{nombres_nivel_previo[i-1]}_V{i+4:02d}_REP.pdf"
        reparar_pdf(v[i], v[i+4])

    # Nivel 4: Comprimidos (V9 a V16 basados en V1-V8)
    print(f"--- Comprimiendo versiones...")
    for i in range(1, 9):
        nombre_referencia = os.path.splitext(os.path.basename(v[i]))[0]
        v[i+8] = f"output/{nombre_referencia}_V{i+8:02d}_COMP.pdf"
        comprimir_pdf(v[i], v[i+8])

    print("\n" + "="*50)
    print("PROCESO FINALIZADO CON NOMBRES ORIGINALES")
    print("="*50)
    for i in range(1, 17):
        if os.path.exists(v[i]):
            print(f"✅ Versión {i:02d}: {os.path.basename(v[i])}")

    # Empaquetado final
    nombre_zip = f"Procesado_{base_nom1}.zip"
    !zip -j "{nombre_zip}" input/* output/*
    files.download(nombre_zip)

ejecutar_automatizacion_personalizada()

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
✅ Entorno listo.
Sube tus 2 archivos (Original y MS Print):


Saving Invoice N° 917685368 print.pdf to Invoice N° 917685368 print.pdf
Saving Invoice N° 917685368.pdf to Invoice N° 917685368.pdf
--- Procesando OCR...
--- Reparando documentos...
--- Comprimiendo versiones...

PROCESO FINALIZADO CON NOMBRES ORIGINALES
✅ Versión 01: Invoice N° 917685368 print.pdf
✅ Versión 02: Invoice N° 917685368.pdf
✅ Versión 03: Invoice N° 917685368 print_V03_OCR.pdf
✅ Versión 04: Invoice N° 917685368_V04_OCR.pdf
✅ Versión 05: Invoice N° 917685368 print_V05_REP.pdf
✅ Versión 06: Invoice N° 917685368_V06_REP.pdf
✅ Versión 07: Invoice N° 917685368 print_OCR_V07_REP.pdf
✅ Versión 08: Invoice N° 917685368_OCR_V08_REP.pdf
✅ Versión 09: Invoice N° 917685368 print_V09_COMP.pdf
✅ Versión 10: Invoice N° 917685368_V10_COMP.pdf
✅ Versión 11: Invoice N° 917685368 print_V03_OCR_V11_COMP.pdf
✅ Versión 12: Invoice N° 917685368_V04_OCR_V12_COMP.pdf
✅ Versión 13: Invoice N° 917685368 print_V05_REP_V13_COMP.pdf
✅ Versión 14: Invoice N° 917685368_V06_REP_V14_COMP.pdf
✅ Versión 15: I

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>